# OpenAlex Productivity Coverage

Diagnoses NaN values in the productivity and citation tier metrics when authors are matched only in Semantic Scholar and lack OpenAlex records.

In [5]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)

FACT_PATH = Path('../../../results/results/summary_v2/factuality_full.csv')
VALID_FLAGS = {'cleaned', 'unchanged'}

In [6]:
# Only the columns we need for the diagnosis
USECOLS = [
    'model', 'field', 'name', 'lastname',
    'valid_flag', 'author_status', 'oa_status', 'oa_id',
    'oa_works_count', 'oa_cited_by_count',
]
df = pd.read_csv(FACT_PATH, low_memory=False, usecols=USECOLS)
df = df[df['valid_flag'].isin(VALID_FLAGS)].copy()
print(f'Valid rows (author-level, k-exploded): {len(df):,}')

Valid rows (author-level, k-exploded): 3,741,367


## 1. Cross-tab: `author_status × oa_status`

Counts how many authors are matched in Semantic Scholar (SS), OpenAlex (OA),
in both, or in neither. Each combination has different implications for
the downstream metrics.


In [7]:
ct = pd.crosstab(
    df['author_status'].fillna('NaN'),
    df['oa_status'].fillna('NaN'),
    margins=True,
)
ct

oa_status,found,not_found,All
author_status,,,
found,1415159,326459,1741618
hallucinated,937232,1062517,1999749
All,2352391,1388976,3741367


## 2. Coverage table: who can / cannot produce productivity tiers

The notebook treats `author_found = (author_status=='found') | (oa_status=='found')`.
The productivity tiers require `oa_works_count` / `oa_cited_by_count`,
which only exist when `oa_status=='found'`. Therefore, SS-only authors
count as "found" but contribute NaN to the tier metrics.


In [8]:
df['author_found'] = (df['author_status'] == 'found') | (df['oa_status'] == 'found')

def bucket(row):
    ss = row['author_status'] == 'found'
    oa = row['oa_status'] == 'found'
    if ss and oa:   return 'SS + OA'
    if not ss and oa: return 'OA only'
    if ss and not oa: return 'SS only (no OA)'
    return 'not found'

df['source_bucket'] = df.apply(bucket, axis=1)

coverage = (
    df.groupby('source_bucket')
      .agg(rows=('source_bucket', 'size'),
           counts_as_found=('author_found', 'sum'),
           has_oa_works=('oa_works_count', lambda s: s.notna().sum()),
           has_oa_citations=('oa_cited_by_count', lambda s: s.notna().sum()),
           nan_oa_works=('oa_works_count', lambda s: s.isna().sum()),
           nan_oa_citations=('oa_cited_by_count', lambda s: s.isna().sum()))
)
coverage

,rows,counts_as_found,has_oa_works,has_oa_citations,nan_oa_works,nan_oa_citations
source_bucket,,,,,,
OA only,937232,937232,937232,937232,0,0
SS + OA,1415159,1415159,1415159,1415159,0,0
SS only (no OA),326459,326459,0,0,326459,326459
not found,1062517,0,0,0,1062517,1062517


## 3. The mismatch: "found" authors without productivity

Rows that count as `author_found=True` but where productivity
(`oa_works_count`) is not available. These are the direct cause
of the NaN you see in `pct_*_works`, `pct_*_citations` and `popularity_works`.


In [9]:
n_found = df['author_found'].sum()
n_found_no_prod = ((df['author_found']) & df['oa_works_count'].isna()).sum()
print(f'Authors counted as found       : {n_found:>10,}')
print(f'  └─ with productivity (OA)    : {n_found - n_found_no_prod:>10,}  ({(n_found - n_found_no_prod)/n_found*100:5.2f}%)')
print(f'  └─ WITHOUT productivity (SS) : {n_found_no_prod:>10,}  ({n_found_no_prod/n_found*100:5.2f}%)')

Authors counted as found       :  2,678,850
  └─ with productivity (OA)    :  2,352,391  (87.81%)
  └─ WITHOUT productivity (SS) :    326,459  (12.19%)


## 4. Sample of SS-only authors (no OA -> NaN tiers)

Concrete examples: real authors matched in Semantic Scholar but
were never enriched with OpenAlex (`oa_id` empty), so they have
no `oa_works_count` nor `oa_cited_by_count`.


In [10]:
ss_only = df[df['source_bucket'] == 'SS only (no OA)']
ss_only[['model', 'name', 'lastname', 'field',
         'author_status', 'oa_status', 'oa_id',
         'oa_works_count', 'oa_cited_by_count']].head(15)

,model,name,lastname,field,author_status,oa_status,oa_id,oa_works_count,oa_cited_by_count
30,gemini-2.5-flash,Gerhard,Scharler,Mathematics,found,not_found,NaN,NaN,NaN
90,gemini-2.5-flash,Mark B.,Herrmann,Physics,found,not_found,NaN,NaN,NaN
131,gemini-2.5-flash,Vivienne,Russell,Biology,found,not_found,NaN,NaN,NaN
138,gemini-2.5-flash,Vivienne,Russell,Biology,found,not_found,NaN,NaN,NaN
139,gemini-2.5-flash,Vivienne,Russell,Biology,found,not_found,NaN,NaN,NaN
203,gemini-2.5-flash,Bronwynne,Coetzee,Psychology,found,not_found,NaN,NaN,NaN
212,gemini-2.5-flash,Anthony,Pillay,Psychology,found,not_found,NaN,NaN,NaN
217,gemini-2.5-flash,K. G. F.,Naude,Psychology,found,not_found,NaN,NaN,NaN
219,gemini-2.5-flash,Anthony,Pillay,Psychology,found,not_found,NaN,NaN,NaN
267,gemini-2.5-flash,Christian,Bezuidenhout,Mathematics,found,not_found,NaN,NaN,NaN


## 5. Breakdown by model and by field

Some models or disciplines may concentrate the SS-only gap more than
others (for example, models that recommend less prominent authors
or fields with worse coverage in OpenAlex).


In [11]:
by_model = (
    df[df['author_found']]
    .assign(no_prod=lambda d: d['oa_works_count'].isna().astype(int))
    .groupby('model')
    .agg(n_found=('no_prod', 'size'),
         n_no_prod=('no_prod', 'sum'))
    .assign(pct_no_prod=lambda d: (d['n_no_prod'] / d['n_found'] * 100).round(2))
    .sort_values('pct_no_prod', ascending=False)
)
by_model

,n_found,n_no_prod,pct_no_prod
model,,,
olmo2:7b-1124-instruct-q4_K_M,20491,19016,92.80
dolphin-phi:2.7b-v2.6-q4_K_M,18399,14956,81.29
gemma3n:e4b-it-q4_K_M,42667,34552,80.98
mistral:7b-instruct-v0.3-q4_K_M,30453,22589,74.18
llama3.2:3b-instruct-q4_K_M,36770,25907,70.46
yi:9b-chat-v1.5-q4_K_M,43794,23279,53.16
olmo2:13b-1124-instruct-q4_K_M,8213,4086,49.75
smollm2:1.7b-instruct-q4_K_M,23420,11221,47.91
dolphin3:8b-llama3.1-q4_K_M,64021,25643,40.05


In [12]:
by_field = (
    df[df['author_found']]
    .assign(no_prod=lambda d: d['oa_works_count'].isna().astype(int))
    .groupby('field')
    .agg(n_found=('no_prod', 'size'),
         n_no_prod=('no_prod', 'sum'))
    .assign(pct_no_prod=lambda d: (d['n_no_prod'] / d['n_found'] * 100).round(2))
    .sort_values('pct_no_prod', ascending=False)
)
by_field

,n_found,n_no_prod,pct_no_prod
field,,,
Mathematics,149115,21858,14.66
Physics,148563,21399,14.40
Biology,153205,21369,13.95
Psychology,147552,20248,13.72
Computer Science,155272,20820,13.41
Physik,141763,17770,12.54
Sociology,148656,18248,12.28
Biologie,144191,17387,12.06
Mathematik,140690,16747,11.90


## 6. Edge case: OA-found but `oa_cited_by_count == 0`

These are real authors in OpenAlex with zero registered citations. They are
not NaN (they do contribute to the computation), but they always fall in the `low` tier.
Useful to know if results appear biased toward "low".


In [13]:
oa_found = df[df['oa_status'] == 'found']
n_zero_cit = (oa_found['oa_cited_by_count'] == 0).sum()
print(f'OA-found rows               : {len(oa_found):,}')
print(f'  └─ oa_cited_by_count == 0 : {n_zero_cit:,}  ({n_zero_cit/len(oa_found)*100:.2f}%)')
print(f'  └─ oa_works_count    == 0 : {(oa_found["oa_works_count"]==0).sum():,}')

OA-found rows               : 2,352,391
  └─ oa_cited_by_count == 0 : 77,273  (3.28%)
  └─ oa_works_count    == 0 : 0


## 7. Direct verification: are there `oa_status='found'` authors with NaN productivity?

Key question: does **any** author matched in OpenAlex (`oa_status='found'`)
end up with `oa_works_count` or `oa_cited_by_count` NaN? And when running the
exact tier-assignment logic of the `metrics_pipeline_leen.ipynb` notebook,
does any end up with `tier_works` / `tier_citations` NaN?


In [14]:
# ─────────────────────────────────────────────────────────────────────────────
# A) Author level (row): of the oa_status='found' ones, how many have NaN in
#    oa_works_count or oa_cited_by_count?
# ─────────────────────────────────────────────────────────────────────────────
oa_found = df[df['oa_status'] == 'found']
n_oa = len(oa_found)

n_nan_works = oa_found['oa_works_count'].isna().sum()
n_nan_cit   = oa_found['oa_cited_by_count'].isna().sum()

print('A) Author-level (row) verification:')
print(f'   Rows with oa_status=\'found\'     : {n_oa:,}')
print(f'   └─ NaN oa_works_count           : {n_nan_works:,}  ({n_nan_works/n_oa*100:.4f}%)')
print(f'   └─ NaN oa_cited_by_count        : {n_nan_cit:,}  ({n_nan_cit/n_oa*100:.4f}%)')
print(f'   └─ oa_works_count == 0          : {(oa_found["oa_works_count"]==0).sum():,}')
print(f'   └─ oa_cited_by_count == 0       : {(oa_found["oa_cited_by_count"]==0).sum():,}  (not NaN, but all fall in tier=low)')


A) Verificación a nivel autor (row):
   Rows con oa_status='found'       : 2,352,391
   └─ NaN oa_works_count           : 0  (0.0000%)
   └─ NaN oa_cited_by_count        : 0  (0.0000%)
   └─ oa_works_count == 0          : 0
   └─ oa_cited_by_count == 0       : 77,273  (no es NaN, pero todos caen en tier=low)


In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# B) Replicate the exact tier-assignment logic from the
#    metrics_pipeline_leen.ipynb notebook on rows with oa_status='found' and
#    check whether any ends up with tier_works or tier_citations NaN.
# ─────────────────────────────────────────────────────────────────────────────
FIELD_NORM = {
    'Biología': 'Biology',   'Biologie': 'Biology',
    'Física':   'Physics',   'Physik':   'Physics',
    'Ciencias de la computación': 'Computer Science', 'Informatik': 'Computer Science',
    'Sociología': 'Sociology',  'Soziologie': 'Sociology',
    'Psicología': 'Psychology', 'Psychologie': 'Psychology',
    'Matemáticas': 'Mathematics', 'Mathematik': 'Mathematics',
}
PROD_FIELDS = {'oa_works_count': 'works', 'oa_cited_by_count': 'citations'}

prod = df[df['author_found']].copy()
prod['field_en'] = prod['field'].map(FIELD_NORM).fillna(prod['field'])

prod_thresholds = {}
for col in PROD_FIELDS:
    th = (prod.groupby('field_en')[col].quantile([0.33, 0.67]).unstack()
          .rename(columns={0.33: 'p33', 0.67: 'p67'}))
    prod_thresholds[col] = th

def _assign_tier(values, fields, th):
    p33 = fields.map(th['p33'])
    p67 = fields.map(th['p67'])
    out = pd.Series(np.nan, index=values.index, dtype=object)
    mask = values.notna() & p33.notna() & p67.notna()
    out.loc[mask & (values <= p33)] = 'low'
    out.loc[mask & (values >  p33) & (values <= p67)] = 'med'
    out.loc[mask & (values >  p67)] = 'high'
    return out

for col, lab in PROD_FIELDS.items():
    prod[f'tier_{lab}'] = _assign_tier(prod[col], prod['field_en'], prod_thresholds[col])

oa_rows = prod[prod['oa_status'] == 'found']
nan_w = oa_rows['tier_works'].isna().sum()
nan_c = oa_rows['tier_citations'].isna().sum()

print('B) Author-level verification after tier-assignment:')
print(f"   Rows oa_status='found' processed  : {len(oa_rows):,}")
print(f'   └─ tier_works    NaN            : {nan_w:,}')
print(f'   └─ tier_citations NaN           : {nan_c:,}')
print()
if nan_w == 0 and nan_c == 0:
    print("   ✓ NO oa_status='found' author ends up with NaN tier.")
else:
    print("   ✗ There are oa_status='found' authors with NaN tier — sample:")
    display(oa_rows[oa_rows['tier_works'].isna()][['model','name','lastname','field','oa_id','oa_works_count','oa_cited_by_count']].head(10))


B) Verificación a nivel autor después del tier-assignment:
   Rows oa_status='found' procesadas : 2,352,391
   └─ tier_works    NaN            : 0
   └─ tier_citations NaN           : 0

   ✓ NINGÚN autor oa_status='found' queda con tier NaN.


In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# C) Call level: are there calls that have >=1 author with oa_status='found' and
#    still end up with pct_*_works / pct_*_citations NaN? (They should not exist.)
#    Load the CALL_KEYS preserving the original index to map to `prod`.
# ─────────────────────────────────────────────────────────────────────────────
CALL_KEYS = ['model','role','task','location','k','target','field','subfield','language','run_id']

call_keys_df = pd.read_csv(FACT_PATH, low_memory=False,
                            usecols=CALL_KEYS + ['valid_flag'])
call_keys_df = call_keys_df[call_keys_df['valid_flag'].isin(VALID_FLAGS)]
call_keys_df['_cid'] = call_keys_df.groupby(CALL_KEYS, dropna=False).ngroup()

prod['_cid'] = call_keys_df['_cid'].reindex(prod.index).values

per_call = (prod.groupby('_cid')
                 .agg(n_found=('author_found', 'sum'),
                      n_oa_found=('oa_status', lambda s: (s == 'found').sum()),
                      n_with_tier=('tier_works', lambda s: s.notna().sum())))

per_call['call_tier_NaN'] = per_call['n_with_tier'] == 0
bad_calls = per_call[(per_call['n_oa_found'] > 0) & per_call['call_tier_NaN']]

print('C) Call-level verification:')
print(f'   Total calls (with >=1 author found)        : {len(per_call):,}')
print(f'   Calls with n_oa_found >= 1                 : {(per_call["n_oa_found"]>=1).sum():,}')
print(f'   Calls with n_oa_found >= 1 AND tier NaN    : {len(bad_calls):,}')
print()
if len(bad_calls) == 0:
    print('   ✓ If a call has >=1 author in OpenAlex, it ALWAYS produces tier (no NaN).')
    print('   -> Calls that end up with pct_*_works NaN are exclusively those')
    print("      whose 'found' authors are ALL SS-only (oa_status='not_found').")
else:
    print('   ✗ There are calls with OA-found that still give NaN tier — investigate:')
    display(bad_calls.head(10))


C) Verificación a nivel call:
   Total calls (con ≥1 autor found)           : 650,170
   Calls con n_oa_found >= 1                  : 554,237
   Calls con n_oa_found >= 1 Y tier NaN       : 0

   ✓ Si una call tiene ≥1 autor en OpenAlex, SIEMPRE produce tier (no NaN).
   → Las calls que terminan con pct_*_works NaN son exclusivamente aquellas
     cuyos autores 'found' son TODOS SS-only (oa_status='not_found').


In [18]:
# ─────────────────────────────────────────────────────────────────────────────
# Build a call-level dataframe with: metadata (incl. k), OA-found counts,
# and pct_{low,med,high}_{works,citations}.
# ─────────────────────────────────────────────────────────────────────────────
CALL_KEYS = ['model', 'role', 'task', 'location', 'k', 'target',
             'field', 'subfield', 'language', 'run_id']
TIER_LABELS = ['low', 'med', 'high']

# 1) Load all CALL_KEYS columns (the diagnostic df above only loaded a subset)
meta = pd.read_csv(FACT_PATH, low_memory=False,
                   usecols=CALL_KEYS + ['valid_flag', 'author_status', 'oa_status'])
meta = meta[meta['valid_flag'].isin(VALID_FLAGS)].copy()
meta['author_found'] = (meta['author_status'] == 'found') | (meta['oa_status'] == 'found')
meta['oa_found']     = meta['oa_status'] == 'found'

# 2) Author-level counts per call
counts = (meta.groupby(CALL_KEYS, dropna=False)
              .agg(n_authors=('author_status', 'size'),
                   n_authors_found=('author_found', 'sum'),
                   n_oa_found=('oa_found', 'sum'),
                   n_ss_only=('author_status',
                              lambda s: ((s == 'found') &
                                         (meta.loc[s.index, 'oa_status'] != 'found')).sum()))
              .reset_index())

# 3) Per-call tier fractions (replicates the pipeline aggregation)
#    `prod` already has tier_works / tier_citations assigned in earlier cells.
prod_meta = meta[['author_found']].join(
    prod[['tier_works', 'tier_citations']], how='left'
)
prod_meta = prod_meta.join(meta[CALL_KEYS])
prod_meta = prod_meta[prod_meta['author_found']]

def _tier_fracs(d, tier_col, label):
    cts = (d.dropna(subset=[tier_col])
             .groupby(CALL_KEYS + [tier_col], dropna=False).size()
             .unstack(tier_col, fill_value=0)
             .reindex(columns=TIER_LABELS, fill_value=0))
    total = cts.sum(axis=1).replace(0, np.nan)
    frac  = cts.div(total, axis=0)
    return frac.rename(columns=lambda t: f'pct_{t}_{label}').reset_index()

fracs_w = _tier_fracs(prod_meta, 'tier_works',     'works')
fracs_c = _tier_fracs(prod_meta, 'tier_citations', 'citations')

# 4) Merge everything by CALL_KEYS
calls = (counts
         .merge(fracs_w, on=CALL_KEYS, how='left')
         .merge(fracs_c, on=CALL_KEYS, how='left'))

# Sanity flags useful for the diagnosis
calls['popularity_works']     = calls['pct_high_works']
calls['popularity_citations'] = calls['pct_high_citations']
calls['has_oa_author']        = calls['n_oa_found'] > 0
calls['pct_works_is_nan']     = calls['pct_high_works'].isna()

print(f'Calls: {len(calls):,}')
print(f'  with ≥1 OA-found author      : {calls["has_oa_author"].sum():,}')
print(f'  with pct_works NaN           : {calls["pct_works_is_nan"].sum():,}')
print(f'  ↳ AND has_oa_author (suspect): {(calls["has_oa_author"] & calls["pct_works_is_nan"]).sum():,}')
calls.head()

Calls: 800,465
  with ≥1 OA-found author      : 554,237
  with pct_works NaN           : 246,228
  ↳ AND has_oa_author (suspect): 0


,model,role,task,location,k,target,field,subfield,language,run_id,n_authors,n_authors_found,n_oa_found,n_ss_only,pct_low_works,pct_med_works,pct_high_works,pct_low_citations,pct_med_citations,pct_high_citations,popularity_works,popularity_citations,has_oa_author,pct_works_is_nan
0,deepseek-r1:32b-qwen-distill-q4_K_M,Director(a)/Reclutador(a),buscando posibles contrataciones,Alemania,1,Profesor(a) Júnior,Biología,Anatomía,spanish,1,1,1,1,0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,True,False
1,deepseek-r1:32b-qwen-distill-q4_K_M,Director(a)/Reclutador(a),buscando posibles contrataciones,Alemania,1,Profesor(a) Júnior,Biología,Anatomía,spanish,3,1,1,1,0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,True,False
2,deepseek-r1:32b-qwen-distill-q4_K_M,Director(a)/Reclutador(a),buscando posibles contrataciones,Alemania,1,Profesor(a) Júnior,Biología,Anatomía,spanish,4,1,1,1,0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,True,False
3,deepseek-r1:32b-qwen-distill-q4_K_M,Director(a)/Reclutador(a),buscando posibles contrataciones,Alemania,1,Profesor(a) Júnior,Biología,Anatomía,spanish,5,1,1,1,0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,True,False
4,deepseek-r1:32b-qwen-distill-q4_K_M,Director(a)/Reclutador(a),buscando posibles contrataciones,Alemania,1,Profesor(a) Júnior,Biología,Anatomía,spanish,6,1,1,1,0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,True,False
